In [71]:
!pip install -q firecrawl-py pandas

In [72]:
from firecrawl import Firecrawl
from pydantic import BaseModel
from typing import List, Optional
from urllib.parse import urljoin

import pandas as pd
import os
import re
import time

firecrawl access through api keys

In [73]:
api_key = input("Enter your Firecrawl API key: ")

app = Firecrawl(api_key=api_key)

print("Firecrawl client is ready")

Enter your Firecrawl API key: fc-9e4a84517381416c83d972fb7df22f5f
Firecrawl client is ready


### Google Drive mount

In [74]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [75]:
FOLDER = "/content/drive/MyDrive/jobdatacsv"
CSV_PATH = f"{FOLDER}/my_jobs.csv"

os.makedirs(FOLDER, exist_ok=True)

print("CSV path:", CSV_PATH)
print("File exists:", os.path.exists(CSV_PATH))

CSV path: /content/drive/MyDrive/jobdatacsv/my_jobs.csv
File exists: True


### Job boards

In [76]:
BOARDS = {
    # Nepal
    "merojob": {
        "url": "https://merojob.com/category/it-telecommunication/",
        "country": "Nepal",
        "work_type": "Onsite",
    },

    "kumarijob": {
        "url": "https://www.kumarijob.com/search?keywords=data",
        "country": "Nepal",
        "work_type": "Onsite",
    },

    "jobsnepal": {
        "url": "https://www.jobsnepal.com/jobs?category=it-computer",
        "country": "Nepal",
        "work_type": "Onsite",
    },

    # Remote
    "remoteok": {
        "url": "https://remoteok.com/remote-data-science-jobs",
        "country": "Anywhere",
        "work_type": "Remote",
    },

    "weworkremotely": {
        "url": "https://weworkremotely.com/categories/remote-data-jobs",
        "country": "Anywhere",
        "work_type": "Remote",
    },

    "himalayas": {
        "url": "https://himalayas.app/jobs/data-science",
        "country": "Anywhere",
        "work_type": "Remote",
    },

    "remote_co": {
        "url": "https://remote.co/remote-jobs/data/",
        "country": "Anywhere",
        "work_type": "Remote",
    },

    "workingnomads": {
        "url": "https://www.workingnomads.com/jobs?category=data",
        "country": "Anywhere",
        "work_type": "Remote",
    },

    # AI company board
    "huggingface": {
        "url": "https://job-boards.greenhouse.io/huggingface",
        "country": "Anywhere",
        "work_type": "Remote",
    },
}

Listing schema

In [77]:
class JobLink(BaseModel):
    title: str
    url: str


class JobLinkList(BaseModel):
    jobs: List[JobLink]

Listing Firecrawl format

In [78]:
listing_format = {
    "type": "json",
    "schema": JobLinkList.model_json_schema(),
    "prompt": (
        "Extract every job opening listed on this page. "
        "For each one give the exact job title and the full URL "
        "of its job page. "
        "Ignore navigation links, categories and adverts."
    )
}

Get listings

In [79]:
def get_listings(board_url):

    page = app.scrape(
        board_url,
        formats=[listing_format]
    )

    jobs = (page.json or {}).get("jobs", [])

    for job in jobs:
        job["url"] = urljoin(
            board_url,
            job["url"]
        )

    # Remove duplicate URLs
    seen = set()
    unique_jobs = []

    for job in jobs:
        if job["url"] not in seen:
            seen.add(job["url"])
            unique_jobs.append(job)

    return unique_jobs

## get_listings() — First Scraping Stage

The `get_listings()` function scrapes a job website's listing page using Firecrawl.

### Steps:

1. Firecrawl opens the job listing page.

2. The `listing_format` tells Firecrawl to extract only:
   - `title`
   - `url`

3. The extracted JSON is stored in `jobs`.

4. `urljoin()` converts relative URLs into complete/absolute URLs.

   Example:
   `/jobs/123`
   →
   `https://example.com/jobs/123`

5. A `set()` called `seen` is used to remember URLs that have already appeared.

6. If a URL has not appeared before, the job is added to `unique_jobs`.

7. If the same URL appears again, it is skipped.

### Important:

This function ONLY finds unique job listings.

It does NOT:
- check whether the job is AI/ML/Data related
- check the CSV
- scrape detailed job information
- add jobs to the CSV
- check the 7-day rule

### Flow:

Website Listing Page
        ↓
Firecrawl
        ↓
title + URL
        ↓
Convert URLs to full URLs
        ↓
Remove duplicate URLs
        ↓
Return unique jobs

Example:

Input:
- ML Engineer → `/jobs/1`
- Data Scientist → `/jobs/2`
- ML Engineer → `/jobs/1`

Output:
- ML Engineer → `https://example.com/jobs/1`
- Data Scientist → `https://example.com/jobs/2`

The duplicate `/jobs/1` is removed.

Relevance filter

In [80]:
PATTERNS = [
    "data scien",
    "data analy",
    "analytics",
    "machine learning",
    "deep learning",
    "applied scientist",
    "research scientist",
    "ai research",
    "research engineer",
    "nlp",
    "computer vision",
    "llm",
]

SHORT_WORDS = ["ml", "ai"]


def is_relevant(title):

    t = title.lower()

    if any(pattern in t for pattern in PATTERNS):
        return True

    return any(
        re.search(rf"\b{word}\b", t)
        for word in SHORT_WORDS
    )

### `is_relevant()`

Checks whether a job title is related to AI/ML/Data.

- Converts the title to lowercase.
- Checks for keywords like:
  `machine learning`, `data science`, `deep learning`, `NLP`, `LLM`, etc.
- Checks `AI` and `ML` as separate words.
- Returns:
  - `True` → relevant job, continue processing.
  - `False` → irrelevant job, discard.

It does NOT add the job to CSV or scrape detailed information.

NEW Job schema

In [81]:
class Job(BaseModel):
    title: str
    company: Optional[str] = None

    # IMPORTANT:
    # Actual date the job was posted
    posted_date: Optional[str] = None

    country: Optional[str] = None
    work_type: Optional[str] = None
    salary: Optional[str] = None

    skills: List[str] = []

    apply_url: Optional[str] = None

### `Job` Schema

Defines the **full details of a job** that will be extracted from the individual job page.

It stores:
- `title` → job title
- `company` → company name
- `posted_date` → actual date the job was posted
- `country` → job location/country
- `work_type` → remote, onsite, hybrid, etc.
- `salary` → salary information
- `skills` → required skills
- `apply_url` → application URL

`Optional` fields can be missing, while `skills` is a list of skills.

This schema is used in the **second Firecrawl scrape**, after the job passes the relevance and CSV URL checks.

Tell Firecrawl to find the REAL posting date

In [82]:
job_format = {
    "type": "json",
    "schema": Job.model_json_schema(),

    "prompt": (
        "Extract the job posting on this page. "

        "For title, give the exact job title. "

        "For company, give the company name. "

        "For posted_date, find the ORIGINAL date when this job "
        "was posted. Return it as YYYY-MM-DD whenever possible. "

        "Do NOT use today's date. "
        "Do NOT use the application deadline. "
        "Do NOT use a page updated date unless the page clearly "
        "identifies it as the original posting date. "

        "If the posting date cannot be found, leave posted_date empty. "

        "For skills, list only concrete technologies, tools and "
        "programming languages mentioned in the requirements. "

        "For country, give the country the job is based in, "
        "or Anywhere if it can be done from any country. "

        "For work_type, answer Remote, Onsite, or Hybrid. "

        "For salary, extract the stated salary if available. "

        "For apply_url, give the actual link used to apply. "

        "Do not invent anything."
    )
}

In [83]:
def load_known_urls(csv_path):

    if os.path.exists(csv_path):

        df = pd.read_csv(csv_path)

        if "url" in df.columns:
            return set(
                df["url"]
                .dropna()
                .astype(str)
            )

    return set()

### `load_known_urls` — The Memory

This function reads the CSV from Google Drive and collects all the job URLs the agent already knows.

- If the file exists → returns a set of known URLs  
- If the file does not exist (first run) → returns an empty set  

This is how the agent avoids scraping the same job twice.

Harvest one board

In [84]:
def harvest_board(
    name,
    cfg,
    known_urls,
    per_board=3
):

    print(f"\n--- {name}")

    # --------------------------------
    # 1. Get listing page
    # --------------------------------

    try:

        listings = get_listings(
            cfg["url"]
        )

    except Exception as e:

        print(
            f"Board failed: {type(e).__name__}"
        )

        return []


    # --------------------------------
    # 2. Keep only AI/ML/Data jobs
    # --------------------------------

    relevant = [
        job
        for job in listings
        if is_relevant(job["title"])
    ]


    # --------------------------------
    # 3. Check CSV memory
    # --------------------------------

    new_jobs = [
        job
        for job in relevant
        if job["url"] not in known_urls
    ]


    print(
        f"Total: {len(listings)} | "
        f"Relevant: {len(relevant)} | "
        f"New: {len(new_jobs)}"
    )


    # --------------------------------
    # 4. Scrape ONLY new jobs
    # --------------------------------

    output = []

    for job in new_jobs[:per_board]:

        try:

            data = app.scrape(
                job["url"],
                formats=[job_format]
            ).json or {}


            # Known by our program
            data["url"] = job["url"]
            data["source"] = name


            # Board defaults
            data["country"] = (
                data.get("country")
                or cfg["country"]
            )

            data["work_type"] = (
                data.get("work_type")
                or cfg["work_type"]
            )


            # Fallback apply URL
            if not data.get("apply_url"):
                data["apply_url"] = job["url"]


            # Important:
            # Do NOT create posted_date = today.
            # We need the REAL posting date.
            if not data.get("posted_date"):

                print(
                    f"    WARNING: no posting date found: "
                    f"{job['title']}"
                )

            output.append(data)

            print(
                f"    OK: {data.get('title')} | "
                f"posted: {data.get('posted_date')}"
            )


        except Exception as e:

            print(
                f"    FAILED: {job['title']} "
                f"({type(e).__name__})"
            )


        time.sleep(1)


    return output

### `harvest_board` — One Board at a Time

This function handles **one job board completely**.

**What it does step by step:**

1. **Gets the listing page**  
   Scrapes all job titles + links from the board.

2. **Filters relevant jobs**  
   Keeps only Data / ML / AI jobs using `is_relevant()`.

3. **Checks memory**  
   Removes jobs whose URL is already in the CSV (so we don’t scrape them again).

4. **Scrapes only new jobs**  
   Visits the detail page of each new job and extracts:
   - title, company, skills, salary
   - country, work_type
   - apply_url
   - posted_date (if available)

5. **Fills missing values**  
   Uses the board’s default country and work_type when the page doesn’t say.

6. **Returns the new jobs**  
   Ready to be added to the CSV.

**Why this design?**
- One dead board doesn’t stop the whole agent
- Only new jobs cost Firecrawl credits
- Defaults fill honest gaps (especially for Nepali boards)

Load memory

In [85]:
known = load_known_urls(CSV_PATH)

print(
    "Jobs already known:",
    len(known)
)

Jobs already known: 8


### `load_known_urls` — The Memory

This function reads the CSV from Google Drive and collects all the job URLs the agent already knows.

- If the file exists → returns a set of known URLs  
- If the file does not exist (first run) → returns an empty set  

This is how the agent avoids scraping the same job twice.

In [ ]:
PER_BOARD = 20

fresh_jobs = []

for name, cfg in BOARDS.items():

    fresh_jobs.extend(
        harvest_board(
            name,
            cfg,
            known,
            per_board=PER_BOARD
        )
    )

print(
    "New jobs found:",
    len(fresh_jobs)
)


--- merojob


Convert new jobs to DataFrame

In [ ]:
df_fresh = pd.DataFrame(fresh_jobs)

Clean posted_date

In [ ]:
if len(df_fresh) > 0:

    df_fresh["posted_date"] = pd.to_datetime(
        df_fresh["posted_date"],
        errors="coerce"
    )

    df_fresh["skills"] = df_fresh["skills"].apply(
        lambda value:
            ", ".join(value)
            if isinstance(value, list)
            else ""
    )

Load old CSV

In [ ]:
if os.path.exists(CSV_PATH):

    df_old = pd.read_csv(
        CSV_PATH,
        parse_dates=["posted_date"]
    )

    print(
        "Old jobs:",
        len(df_old)
    )

else:

    df_old = pd.DataFrame()

    print("No existing CSV")

In [ ]:
df_all = pd.concat(
    [df_old, df_fresh],
    ignore_index=True
)

Remove duplicate URLs

In [ ]:
if len(df_all) > 0:

    df_all = df_all.drop_duplicates(
        subset=["url"],
        keep="first"
    )

In [ ]:
MAX_AGE_DAYS = 7

now = pd.Timestamp.now().normalize()

df_all["posted_date"] = pd.to_datetime(
    df_all["posted_date"],
    errors="coerce"
)

age_days = (
    now - df_all["posted_date"]
).dt.days

Remove jobs older than 7 days

In [ ]:
has_date = df_all["posted_date"].notna()

expired = (
    has_date
    &
    (age_days > MAX_AGE_DAYS)
)

df_all = df_all[~expired]

In [ ]:
expired = (
    df_all["posted_date"].notna()
    &
    (age_days > MAX_AGE_DAYS)
)

removed = int(expired.sum())

df_all = df_all[~expired]

In [ ]:
COLUMNS = [
    "title",
    "company",
    "posted_date",
    "skills",
    "salary",
    "country",
    "work_type",
    "apply_url",
    "source",
    "url"
]

existing_columns = [
    column
    for column in COLUMNS
    if column in df_all.columns
]

df_save = df_all[existing_columns]

In [ ]:
if len(df_save) > 0:

    df_save.to_csv(
        CSV_PATH,
        index=False
    )

    print(
        f"Saved {len(df_save)} jobs"
    )

else:

    if os.path.exists(CSV_PATH):

        os.remove(CSV_PATH)

        print(
            "All jobs expired. CSV deleted."
        )

Final complete agent function